# Quarterly recommendation splits and a LightGBM ranker

Run `scripts/download_data.py`, then `scripts/make_splits.py` once before this notebook.
We learn from earlier **history → next-quarter outcomes** tasks, then predict the next quarter.
Initial training tasks cover 2003–2006, validation 2007–2010, and test 2011–2015; pre-2003 reviews remain history.
This mirrors `rolling_popularity.ipynb`, with LightGBM in place of the popularity baseline. Popularity is rerun alongside it for comparison.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from remy.splits import RollingSplits, SPLIT_DIR
from remy.model.popularity import Popularity
from remy.model.lightgbm_ranker import LightGBMRanker

splits = RollingSplits()
split = "val"  # Use "test" only after choosing and fixing model settings.

# Model settings: keep these fixed while evaluating a split.
n_train_quarters = 8     # most recent completed tasks to train on (None = all)
n_eval_quarters = None   # score only the first N quarters of the split (None = all), for quick runs
model_kwargs = dict(
    n_popular=200,         # candidates per user: all-time most reviewed
    n_trending=100,        # candidates per user: most reviewed in the last trending_days
    trending_days=90,
    max_train_users=3000,  # users sampled per training task (None = all)
    num_boost_round=200,
    seed=0,
)

splits.windows.groupby("split", sort=False)[
    ["outcome_reviews", "positive_reviews", "positive_users"]
].sum().rename(columns={"positive_users": "positive_user_quarters"})

## Get training data and evaluation data

`splits.training_quarters("2007-01-01")` returns the 16 completed training tasks from 2003–2006. For each task, build features from its `history` and labels from its `outcomes`.
`splits.evaluation_quarters("val")` returns the 16 held-out tasks from 2007–2010. Fit using only the past at each task's `start`, then score against its `outcomes`. Both return ordinary lists: `quarters[0]` gets the first task.

LightGBM learns from these tasks: for each of the last `n_train_quarters` completed tasks it builds candidate recipes per user from that task's `history`, and labels them with that task's `outcomes` (5-star = 1). Earlier evaluation quarters become training tasks automatically once their outcomes are complete. Popularity ignores `training_quarters`; we pass them anyway to illustrate the shared model interface.

## Score, then roll forward

Positives are 5-star reviews in the following quarter, from users with at least three prior reviews of **any** rating, on available, previously unseen recipes. Users with no positive that quarter receive no ranking score.
We compute **full-ranking NDCG** (no cutoff), plus hit rate and recall at 10 and 100; average over users within each quarter, then over quarters equally.

This calculation generalizes the Popularity one. LightGBM only scores a short candidate list per user; every other recipe is implicitly ranked *after* the candidates in popularity order, skipping recipes the user has already seen. A positive outside the candidate list therefore gets an exact rank: `len(candidates) + popularity rank - (seen or candidate recipes ranked ahead of it)`. Popularity is the special case with zero candidates, so one function scores both models and avoids constructing thousands of full catalog lists.

In [ ]:
def evaluate_quarter(recs, ranking, seen, quarter, ks=(10, 100)):
    """Full NDCG, HR@k and Recall@k from exact ranks.

    recs:    {user_id: [recipe_id, ...]} ordered model-scored candidates (empty for Popularity).
    ranking: all available recipe IDs in popularity order (the fallback ordering).
    seen:    {user_id: set(recipe_id)} previously reviewed recipes.
    """
    positives = quarter.outcomes[quarter.outcomes["rating"] == 5]
    global_rank = {recipe: rank for rank, recipe in enumerate(ranking, start=1)}
    scores = []
    for user_id, rows in positives.groupby("user_id"):
        relevant = list(set(rows["recipe_id"]))
        cands = recs.get(user_id, [])
        cand_pos = {recipe: i for i, recipe in enumerate(cands, start=1)}
        # Fallback ordering: popularity, minus anything already seen or already a candidate.
        blocked = set(seen.get(user_id, ())) | set(cands)
        blocked_ranks = np.sort([global_rank[r] for r in blocked if r in global_rank])
        ranks = np.empty(len(relevant), dtype=float)
        for j, recipe in enumerate(relevant):
            if recipe in cand_pos:
                ranks[j] = cand_pos[recipe]
            else:
                pr = global_rank[recipe]
                ranks[j] = len(cands) + pr - np.searchsorted(blocked_ranks, pr)
        dcg = (1 / np.log2(ranks + 1)).sum()
        ideal = (1 / np.log2(np.arange(2, len(relevant) + 2))).sum()
        metrics = {"NDCG": dcg / ideal}
        for k in ks:
            hits = (ranks <= k).sum()
            metrics[f"HR@{k}"] = float(hits > 0)
            metrics[f"Recall@{k}"] = hits / len(relevant)
        scores.append(metrics)
    if not scores:
        raise ValueError(f"No scorable users at {quarter.start.date()}")
    result = {
        "start": quarter.start,
        "eligible_users": len(quarter.users),
        "scored_users": len(scores),
        "positives": len(positives),
    }
    result.update(pd.DataFrame(scores).mean().to_dict())
    return result


def candidate_recall(recs, quarter):
    """Share of positives inside the candidate lists: the ceiling for the reranker at small k."""
    positives = quarter.outcomes[quarter.outcomes["rating"] == 5]
    return float(np.mean([r in set(recs.get(u, ())) for u, r in zip(positives["user_id"], positives["recipe_id"])]))

In [ ]:
rows = []
eval_quarters = splits.evaluation_quarters(split)
if n_eval_quarters is not None:
    eval_quarters = eval_quarters[:n_eval_quarters]

for quarter in eval_quarters:
    train_quarters = splits.training_quarters(quarter.start)
    positive_users = quarter.outcomes.loc[quarter.outcomes["rating"] == 5, "user_id"].unique()

    model = LightGBMRanker(n_train_quarters=n_train_quarters, **model_kwargs)  # Fresh fit each quarter; keep the settings fixed.
    model.fit(
        quarter.history,
        recipes=quarter.recipes,
        training_quarters=train_quarters,
        asof=quarter.start,
    )
    # Current outcomes are used only for scoring, after the fit.
    recs = model.recommend_many(positive_users)
    row = evaluate_quarter(recs, model.ranking_, model.seen_, quarter)
    row.update(model="LightGBM", candidate_recall=candidate_recall(recs, quarter))
    rows.append(row)

    pop = Popularity().fit(quarter.history, recipes=quarter.recipes, training_quarters=train_quarters)
    row = evaluate_quarter({}, pop.ranking_, pop.seen_, quarter)
    row.update(model="Popularity")
    rows.append(row)

results = pd.DataFrame(rows)
results.to_csv(SPLIT_DIR / f"lightgbm_{split}.csv", index=False)
metric_cols = ["NDCG", "HR@10", "HR@100", "Recall@10", "Recall@100"]
summary = results.groupby("model")[metric_cols].mean().T[["Popularity", "LightGBM"]]
summary["Relative lift"] = summary["LightGBM"] / summary["Popularity"] - 1
summary.rename_axis(None, axis=1).round(4)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for name, grp in results.groupby("model"):
    axes[0].plot(grp["start"], grp["NDCG"], marker="o", label=name)
axes[0].set(title=f"LightGBM vs Popularity: rolling {split} performance", ylabel="Mean full-ranking NDCG")
axes[0].set_ylim(bottom=0)
axes[0].legend()
pop_rows = results[results["model"] == "Popularity"]
axes[1].plot(pop_rows["start"], pop_rows["scored_users"], marker="o")
axes[1].set(ylabel="Scored users")
axes[1].set_ylim(bottom=0)
lgb_rows = results[results["model"] == "LightGBM"]
axes[2].plot(lgb_rows["start"], lgb_rows["candidate_recall"], marker="o", color="tab:green")
axes[2].set(ylabel="Candidate recall", xlabel="Quarter start")
axes[2].set_ylim(0, 1)
fig.tight_layout()
plt.show()

The popularity notebook saw first-quarter increases and suggested winter-demand seasonality as a possible, unproven explanation. Compare the two NDCG lines above to see whether LightGBM follows the same pattern or the trending candidates and features shift it.

Candidate recall is the share of positives that make it into the candidate lists. It caps how much the reranker can help at small cutoffs, so if it is low, widen `n_popular` / `n_trending` before tuning the model.

These scores describe active, eligible users, not every visitor; the scored-users panel shows how that population changes. Below are the most-reviewed recipes at the **last checkpoint** (the all-time candidate source, before each user's seen recipes are removed) and what the model relied on.

In [ ]:
top = model.scores_.head(10).sort_values()
names = quarter.recipes.set_index("id")["name"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(names.reindex(top.index), top.values)
ax.set(title=f"Most-reviewed recipes before {quarter.start.date()}", xlabel="Past reviews")
fig.tight_layout()
plt.show()

In [ ]:
imp = model.feature_importance("gain").sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(imp.index, imp.values)
ax.set(title=f"LightGBM feature importance (gain), checkpoint {quarter.start.date()}", xlabel="Total gain")
fig.tight_layout()
plt.show()